# DICE — Notebook 1.2 : scénarios et forçage — Version étudiante


**Objectifs pédagogiques — 1 h 30**
1. Charger une version pédagogique de DICE et repérer ses blocs : paramètres, états initiaux, économie et climat.
2. Construire des scénarios SSP en modifiant les variables exogènes.
3. Combiner les récits SSP avec plusieurs trajectoires de réduction des émissions et de forçage.

> Ce notebook utilise le module local `DICE.py`.


# 0) Charger DICE et simuler la référence


In [ ]:
# Exécutez cette cellule une fois pour vérifier/installer les paquets Python requis pour ce portable.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
# 1) Charger le DICE et l'exécution de base
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
) 

p = Params()
path = init_states(p)
path[:, p.i_mu] = 0.03          # baseline abatement path
path = update_path(path, range(1, p.nT), p)
df_base = mat_to_df(path, p)
df_base.head()


## 1) Scénarios SSP

Nous comparons deux récits extrêmes — SSP1 et SSP5 — à la référence SSP2.

- **SSP1 — Durabilité.** Le monde suit une trajectoire plus soutenable, avec davantage d’éducation et de santé, moins d’inégalités et une consommation moins intensive en matières et en énergie.
- **SSP2 — Voie médiane.** Les tendances sociales, économiques et technologiques prolongent globalement leur évolution actuelle.
- **SSP5 — Développement fondé sur les énergies fossiles.** La croissance rapide repose sur les combustibles fossiles et des modes de vie énergivores ; les émissions restent élevées malgré une forte capacité d’adaptation technologique.

| Paramètre | Symbole | SSP1 — optimiste | SSP2 — référence | SSP5 — pessimiste |
|---|---:|---:|---:|---:|
| Intensité carbone initiale | $\sigma_0$ | −25 % | valeur étalonnée | +25 % |
| Rythme de décarbonation | $g_\sigma$ | baisse 25 % plus rapide | valeur étalonnée | baisse 50 % plus lente |
| Population de long terme | $L_\infty$ | 9 milliards | valeur étalonnée | 16 milliards |
| Croissance démographique | $l_g$ | −20 % | valeur étalonnée | +30 % |
| Productivité initiale | $A_0$ | +5 % | valeur étalonnée | −5 % |
| Croissance de la productivité | $g_A$ | +15 % | valeur étalonnée | −15 % |

Pour chaque scénario, créez un nouvel objet de paramètres, simulez le modèle et comparez production, émissions, dommages et températures.


#### 1-A) Partir de SSP1 et SSP2, puis construire SSP5

Les étalonnages SSP1 et SSP2 sont fournis. Complétez uniquement SSP5 à partir du tableau précédent.


In [ ]:
# Construisez des variantes de paramètres à partir du tableau d'étalonnage ci-dessus.
def make_params_variant(**kwargs):
    q = Params()
    for name, value in kwargs.items():
        setattr(q, name, value)
    return q

# SSP2 : étalonnage de référence.
p_ssp2 = Params()

# SSP1 : durabilité (fourni).
p_ssp1 = make_params_variant(
    sigma0=0.75 * p_ssp2.sigma0,
    gsig=1.25 * p_ssp2.gsig,
    lg=0.80 * p_ssp2.lg,
    Linf=9000,
    A0=1.05 * p_ssp2.A0,
    gA=1.15 * p_ssp2.gA,
)

# À vous : construisez SSP5 en remplaçant les six ellipses.
# Le notebook s'exécute tant que SSP5 n'est pas encore simulé.
p_ssp5 = make_params_variant(
    sigma0=...,
    gsig=...,
    lg=...,
    Linf=...,
    A0=...,
    gA=...,
)


#### 1-B) Simuler et tracer SSP2, puis ajouter SSP1 et SSP5

Le code ci-dessous résout et représente SSP2. Reproduisez ensuite les mêmes étapes pour SSP1 et SSP5, puis superposez leurs trajectoires.


In [ ]:
def simulate_scenario(p, saving_rate=0.20, abatement_rate=0.03):
    sim = init_states(p)
    sim[1:, p.i_s] = saving_rate
    sim[1:, p.i_mu] = abatement_rate
    return update_path(sim, range(1, p.nT), p)

def damage_share_pct(sim, p):
    # La fonction de dommage utilise la température de la période précédente.
    temperature_lag = np.r_[p.T_AT0, sim[:-1, p.i_T_AT]]
    damages = p.a2 * temperature_lag**p.a3
    damages += np.where(temperature_lag > p.a6, p.a4 * temperature_lag**p.a5, 0.0)
    return 100 * damages

# Exemple entièrement résolu : SSP2.
sim_ssp2 = simulate_scenario(p_ssp2)
timevec = range(1, p_ssp2.nT)
years = sim_ssp2[:, p_ssp2.i_time]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes[0, 0].plot(years[1:], sim_ssp2[1:, p_ssp2.i_E], label="SSP2")
axes[0, 0].set_ylabel(f"GtC par pas de {p_ssp2.Delta} ans")
axes[0, 0].set_title("Émissions")
axes[0, 1].plot(years[1:], sim_ssp2[1:, p_ssp2.i_Y], label="SSP2")
axes[0, 1].set_ylabel("Milliers de milliards USD/an")
axes[0, 1].set_title("Production brute")
axes[1, 0].plot(years[1:], damage_share_pct(sim_ssp2, p_ssp2)[1:], label="SSP2")
axes[1, 0].set_ylabel("% de la production brute")
axes[1, 0].set_title("Dommages climatiques")
axes[1, 1].plot(years[1:], sim_ssp2[1:, p_ssp2.i_T_AT], label="SSP2")
axes[1, 1].set_ylabel("°C par rapport au préindustriel")
axes[1, 1].set_title("Température atmosphérique")
for ax in axes.flat:
    ax.grid(alpha=0.25)
    ax.legend()
    ax.set_xlabel("Année")
fig.tight_layout()

# À vous : calculez sim_ssp1 et sim_ssp5 avec simulate_scenario,
# puis ajoutez leurs courbes aux quatre graphiques.
# sim_ssp1 = ...
# sim_ssp5 = ...

# Alias conservé pour les exercices d'atténuation ci-dessous.
sim_base = sim_ssp2
p_base = p_ssp2


#### 1-C) Quels paramètres des récits SSP déterminent le plus le changement climatique ? Comparez les températures des scénarios extrêmes.


> Vous avez écrit la réponse ici.

## 2) Calendriers d’atténuation : atteindre presque zéro émission en 2100, 2080 ou 2060

Les scénarios précédents modifiaient seulement la productivité, la population et l’intensité carbone. Nous introduisons maintenant le taux de réduction des émissions $\mu_t$, qui mesure la part des émissions industrielles évitées.

- Dans la référence, $\mu_t$ reste faible et constant, par exemple 3 %.
- Ici, nous imposons explicitement une trajectoire $\{\mu_t\}$.
- Lorsque $\mu_t\to1$, presque toutes les émissions industrielles sont évitées ; les émissions liées à l’usage des sols restent exogènes.

Dans les scénarios du GIEC, un récit SSP peut être combiné à plusieurs ambitions climatiques. Une transition ambitieuse réduit rapidement les émissions ; une transition tardive reporte le zéro net. Les niveaux de forçage en 2100 — 2,6, 4,5 ou 8,5 W/m² — distinguent notamment SSP1-2.6, SSP2-4.5 et SSP5-8.5.

Interprétez les résultats en opposant décarbonation précoce et action retardée.


#### 2-A) Construire les trajectoires de $\mu_t$

La fonction fournie construit des rampes linéaires atteignant $\mu_t=1$ en 2100, 2080 et 2060. Le taux d’épargne reste fixé à $s_t=0{,}2$. Les états et contrôles sont regroupés dans la matrice $w=[y,x,z]$.


In [ ]:
# Rampes basées sur la date : le code reste valide si Delta change.
def abatement_ramp(années, target_year):
    ramp = np.zeros(len(années))
    target = int(np.argmin(np.abs(années - target_year)))
    ramp[1:target + 1] = np.linspace(0.0, 1.0, target)
    ramp[target + 1:] = 1.0
    return ramp

années = sim_base[:, p_base.i_time]
ramp2060 = abatement_ramp(années, 2060)
ramp2080 = abatement_ramp(années, 2080)
ramp2100 = abatement_ramp(années, 2100)

# Vérifications : chaque trajectoire atteint bien 100 % à la date cible.
for ramp, target_year in [(ramp2060, 2060), (ramp2080, 2080), (ramp2100, 2100)]:
    target = int(np.argmin(np.abs(années - target_year)))
    assert np.isclose(ramp[target], 1.0)

plt.figure(figsize=(8, 4))
for ramp, label in [(ramp2060, "2060"), (ramp2080, "2080"), (ramp2100, "2100")]:
    plt.plot(années[1:], ramp[1:], label=f"Zéro net en {label}")
plt.xlabel("Année")
plt.ylabel(r"Taux d'atténuation $\mu_t$")
plt.ylim(-0.02, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()


#### 2-B) Simuler 2060, puis compléter 2080 et 2100

Comparez le forçage $F_t$, la température atmosphérique $T_{AT,t}$ et les émissions $E_t$. Après modification des contrôles $z$ dans $w$, résolvez à nouveau
$$
y_t=f_p(y_{t-1},x_{t-1},z_t).
$$

Dans quelle mesure la politique climatique réduit-elle les émissions ?


In [ ]:
# Exemple entièrement résolu : zéro net en 2060 sous SSP2.
sim2060 = sim_base.copy()
sim2060[:, p_base.i_mu] = ramp2060
sim2060 = update_path(sim2060, timevec, p_base)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
variables = [p_base.i_F, p_base.i_T_AT, p_base.i_E]
titles = ["Forçage radiatif", "Température atmosphérique", "Émissions"]
units = ["W/m²", "°C", f"GtC par pas de {p_base.Delta} ans"]
for ax, variable, title, unit in zip(axes, variables, titles, units):
    ax.plot(années[1:], sim_base[1:, variable], label="SSP2 — référence")
    ax.plot(années[1:], sim2060[1:, variable], label="SSP2 — zéro net 2060")
    ax.set_title(title)
    ax.set_xlabel("Année")
    ax.set_ylabel(unit)
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()

# À vous : construisez sim2080 et sim2100 avec les mêmes trois étapes,
# puis ajoutez leurs trajectoires aux graphiques.
# sim2080 = ...
# sim2100 = ...


> Vous avez écrit la réponse ici.

### 2-C) Quel est le coût économique de l’atténuation ?

Le code fourni trace pour 2060 :
1. l’écart de consommation $100\times(c_t^{alt}/c_t^{ref}-1)$ ;
2. l’écart de température $T_t^{alt}-T_t^{ref}$.

Reproduisez ensuite l’analyse pour 2080 et 2100. Commentez la perte de consommation pendant la transition et expliquez son origine.


In [ ]:
# Exemple entièrement résolu : coût et bénéfice climatique du zéro net en 2060.
gapC_2060 = 100 * (sim2060[1:, p_base.i_C] / sim_base[1:, p_base.i_C] - 1)
gapT_2060 = sim2060[1:, p_base.i_T_AT] - sim_base[1:, p_base.i_T_AT]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
axes[0].plot(années[1:], gapC_2060, color="tab:red")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Coût en consommation — zéro net 2060")
axes[0].set_ylabel("Écart à SSP2 (%)")
axes[1].plot(années[1:], gapT_2060, color="tab:blue")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Bénéfice climatique — zéro net 2060")
axes[1].set_ylabel("Écart à SSP2 (°C)")
for ax in axes:
    ax.set_xlabel("Année")
    ax.grid(alpha=0.25)
fig.tight_layout()

# À vous : reproduisez ce calcul et ce graphique pour sim2080 et sim2100.


> Vous avez écrit la réponse ici.

## 3) Combiner récits SSP et objectifs de forçage

Un scénario comme SSP1-2.6 associe le récit SSP1 à un forçage radiatif de 2,6 W/m² en 2100.

|      | **1,9** | **2,6** | **4,5** | **7,0** | **8,5** |
|------|---------|---------|---------|---------|---------|
| **SSP1** | SSP1-1.9 | SSP1-2.6 | – | – | – |
| **SSP2** | – | SSP2-2.6 | SSP2-4.5 | SSP2-7.0 | – |
| **SSP3** | – | – | SSP3-4.5 | SSP3-7.0 | SSP3-8.5 |
| **SSP4** | – | SSP4-2.6 | SSP4-4.5 | SSP4-7.0 | – |
| **SSP5** | – | – | SSP5-4.5 | – | SSP5-8.5 |

### Ordres de grandeur du réchauffement en 2100
- **1,9 W/m²** : environ 1,5 °C ;
- **2,6 W/m²** : environ 2 °C ;
- **4,5 W/m²** : environ 2,5 à 3 °C ;
- **7,0 W/m²** : environ 3,5 à 4 °C ;
- **8,5 W/m²** : environ 4,5 °C ou davantage.

Ces valeurs sont des estimations centrales approximatives issues du GIEC AR6. Les faibles forçages exigent une atténuation précoce ; les forts forçages correspondent à une utilisation prolongée des énergies fossiles.


#### 3-A) Pourquoi certaines combinaisons éloignées de la diagonale, comme SSP1-8.5, ne sont-elles généralement pas évaluées ?


> Vous avez écrit la réponse ici.

#### 3-B) Simuler SSP1-1.9 : scénario optimiste et zéro net vers 2060. L’objectif est-il atteignable ?

Les étapes 1 à 4 sont fournies. À vous d’effectuer l’étape 5 : comparer les trajectoires et conclure sur l’objectif de température.


In [ ]:
# Étapes 1 à 4 : construire SSP1 avec une trajectoire de zéro net en 2060.
sim2060_ssp1 = init_states(p_ssp1)                         # 1. initialiser SSP1
sim2060_ssp1[:, p_ssp1.i_s] = sim_base[:, p_base.i_s]      # 2. reprendre l'épargne de référence
sim2060_ssp1[:, p_ssp1.i_mu] = ramp2060                    # 3. imposer la rampe 2060
sim2060_ssp1 = update_path(                                 # 4. simuler la trajectoire
    sim2060_ssp1, range(1, p_ssp1.nT), p_ssp1
)

# Étape 5 — à vous :
# - comparez sim2060_ssp1 à sim_ssp2 pour F, T_AT et E ;
# - relevez la température atmosphérique en 2100 ;
# - concluez : l'objectif d'environ 1,5 °C associé à SSP1-1.9 est-il atteint ?


> Vous avez écrit la réponse ici.

#### 3-C) Peut-on atteindre 4 °C sous SSP1 ? Simulez SSP1 sans atténuation, $\mu_t=0$, pour approcher SSP1-8.5.


In [ ]:
# SSP1-like / no-mitigation stress test:
# démarrer à partir d'init states(p opt), réutiliser le chemin de sauvegarde de base,
# définir la réduction à zéro, puis appeler update_path avec p_opt.
# Comparer avec sim base et rapporter le réchauffement atmosphérique en 2100.


> Vous avez écrit la réponse ici.

#### 3-D) Peut-on limiter le réchauffement à 1,5 °C sous SSP5 ? Faites atteindre $\mu_t=1$ en 2060 pour approcher SSP5-1.9.


In [ ]:
# SSP5-like / strong-mitigation stress test:
# démarrer à partir d'init states(p pes), réutiliser le chemin de sauvegarde de base,
# affecter la rampe 2060, puis appeler update_path avec p_pes.
# Comparer avec sim base et rapporter le réchauffement atmosphérique en 2100.


> Vous avez écrit la réponse ici.